# Reliable DFU Framework — grouped CV, calibration and selective prediction
Runs 5 duplicate-group-safe outer folds × 3 fixed seeds for ConvNeXtV2-Tiny, MobileNetV3-Large and DenseNet121. Primary Drive keeps every full checkpoint. A storage-aware compact secondary backup keeps the active resume state and completed portable checkpoints without duplicating the entire run.

In [ ]:
import os, sys, shutil, subprocess, json, time, uuid
from pathlib import Path
from google.colab import drive

MOUNT=Path('/content/drive'); MY=MOUNT/'MyDrive'
if not MY.is_dir(): drive.mount(str(MOUNT), force_remount=False)
if not MY.is_dir(): raise RuntimeError('Google Drive unavailable; training not started.')
probe=MY/'DFU-ImageGuard'/'_mount_verification'/f'{uuid.uuid4().hex}.txt'
probe.parent.mkdir(parents=True,exist_ok=True); probe.write_text(str(time.time_ns()))
assert probe.read_text(); probe.unlink()
print('Drive write/read verification: PASS')

subprocess.run([sys.executable,'-m','pip','install','-q','timm>=1.0.9','kagglehub>=0.3','ImageHash>=4.3','scikit-learn>=1.5','scipy>=1.13','matplotlib>=3.9','pandas>=2.2','Pillow>=10.4','tabulate>=0.9'],check=True)
REPO='https://github.com/AzizulHakim00/DFU-ImageGuard.git'; BRANCH='reliable-dfu-framework-v1'; WORK=Path('/content/DFU-ImageGuard-reliable')
if WORK.exists(): shutil.rmtree(WORK)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REPO,str(WORK)],check=True)
os.chdir(WORK); sys.path.insert(0,str(WORK))
for module_name in list(sys.modules):
    if module_name=='src' or module_name.startswith('src.'):
        del sys.modules[module_name]
commit=subprocess.run(['git','rev-parse','HEAD'],capture_output=True,text=True,check=True).stdout.strip()
print('Loaded reliable framework commit:',commit)
from src.reliable_runner import ReliableSettings, run_reliable_framework
settings=ReliableSettings(run_id='RELIABLE_DFU_CV_V1',seeds=(2026,2027,2028),folds=(0,1,2,3,4),models=('convnextv2_tiny','mobilenetv3_large','densenet121'),max_epochs=30,patience=7,batch_size=16,num_workers=2,target_sensitivity=.95)
result=run_reliable_framework(settings)
print(json.dumps(result,indent=2,default=str))
